# Error Analysis — BERT vs Gemini

Deep-dive into where BERT and Gemini agree, disagree, and fail.

**Sections:**
1. Load predictions (BERT logits + Gemini JSON)
2. Align samples on shared test IDs
3. Per-class F1 comparison heatmap
4. Disagreement analysis
5. Error categorisation: FP / FN per class
6. Top-20 confused emotion pairs
7. Qualitative examples: BERT✓ LLM✗ and BERT✗ LLM✓

## 1. Imports & Load Predictions

In [ ]:
import sys
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import torch

ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import EMOTION_NAMES, NUM_LABELS
from src.evaluate import per_class_f1, compute_metrics

RESULTS_DIR   = ROOT / "results"
PLOTS_DIR     = RESULTS_DIR / "plots"
BERT_DIR      = RESULTS_DIR / "models" / "bert_base"
GEMINI_DIR    = RESULTS_DIR / "llm" / "gemini_zeroshot"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)
print("Paths:")
print(f"  BERT logits : {BERT_DIR / 'test_logits.npy'}")
print(f"  Gemini preds: {GEMINI_DIR / 'predictions.json'}")

In [ ]:
# ── Load BERT predictions ──────────────────────────────────────
bert_logits_path = BERT_DIR / "test_logits.npy"
bert_labels_path = BERT_DIR / "test_labels.npy"

if not bert_logits_path.exists():
    raise FileNotFoundError(
        f"{bert_logits_path} not found.\n"
        "Run: python -m src.train --config configs/bert_base.yaml first."
    )

bert_logits = torch.tensor(np.load(bert_logits_path))
bert_labels = torch.tensor(np.load(bert_labels_path))

bert_probs = torch.sigmoid(bert_logits).numpy()
bert_preds = (bert_probs >= 0.5).astype(int)
bert_true  = bert_labels.numpy().astype(int)

print(f"BERT logits shape : {bert_logits.shape}")
print(f"BERT labels shape : {bert_labels.shape}")

In [ ]:
# ── Load Gemini predictions ────────────────────────────────────
gemini_pred_path = GEMINI_DIR / "predictions.json"

if not gemini_pred_path.exists():
    raise FileNotFoundError(
        f"{gemini_pred_path} not found.\n"
        "Run: python -m src.llm_inference --config configs/gemini_zeroshot.yaml first."
    )

with open(gemini_pred_path, "r", encoding="utf-8") as f:
    gemini_records = json.load(f)

name_to_idx = {name: i for i, name in enumerate(EMOTION_NAMES)}

def labels_to_multihot(label_list):
    vec = np.zeros(NUM_LABELS, dtype=int)
    for lbl in label_list:
        if lbl in name_to_idx:
            vec[name_to_idx[lbl]] = 1
    return vec

gemini_true_arr  = np.array([labels_to_multihot(r["true_labels"]) for r in gemini_records])
gemini_pred_arr  = np.array([labels_to_multihot(r["predicted_labels"]) for r in gemini_records])
gemini_texts     = [r["text"] for r in gemini_records]
gemini_ids       = [r["id"] for r in gemini_records]

print(f"Gemini records   : {len(gemini_records)}")
print(f"gemini_true shape: {gemini_true_arr.shape}")

## 2. Align on Shared Samples

> BERT is evaluated on the **full** test set; Gemini on a random **2000-sample subset**.  
> For head-to-head comparison we restrict BERT results to the same 2000 IDs.

In [ ]:
# Try to load BERT per-sample IDs (saved by train.py)
bert_ids_path = BERT_DIR / "test_ids.json"

if bert_ids_path.exists():
    with open(bert_ids_path) as f:
        bert_ids = json.load(f)

    shared_id_set = set(gemini_ids) & set(bert_ids)
    print(f"Shared test IDs: {len(shared_id_set)}")

    bert_id_to_idx = {sid: i for i, sid in enumerate(bert_ids)}
    gemini_id_to_idx = {sid: i for i, sid in enumerate(gemini_ids)}

    shared_ids = [sid for sid in gemini_ids if sid in shared_id_set]
    bert_preds_aligned  = np.array([bert_preds[bert_id_to_idx[sid]] for sid in shared_ids])
    bert_true_aligned   = np.array([bert_true[bert_id_to_idx[sid]]  for sid in shared_ids])
    gemini_preds_aligned = np.array([gemini_pred_arr[gemini_id_to_idx[sid]] for sid in shared_ids])
    true_aligned         = bert_true_aligned  # same ground-truth
    texts_aligned        = [gemini_texts[gemini_id_to_idx[sid]] for sid in shared_ids]
else:
    print("test_ids.json not found — using Gemini subset for both models.")
    print("Note: BERT and Gemini may not be evaluated on exactly the same samples.")
    # Fall back: compare on Gemini subset using Gemini ground-truth for both
    bert_preds_aligned   = bert_preds[:len(gemini_records)]
    bert_true_aligned    = bert_true[:len(gemini_records)]
    gemini_preds_aligned = gemini_pred_arr
    true_aligned         = gemini_true_arr
    texts_aligned        = gemini_texts

N_shared = len(true_aligned)
print(f"Analysis samples: {N_shared}")

## 3. Per-class F1 Comparison Heatmap

In [ ]:
from sklearn.metrics import f1_score

bert_f1_per_class   = f1_score(true_aligned, bert_preds_aligned,   average=None, zero_division=0)
gemini_f1_per_class = f1_score(true_aligned, gemini_preds_aligned, average=None, zero_division=0)
diff_f1 = bert_f1_per_class - gemini_f1_per_class

# Build DataFrame for heatmap
df_f1 = pd.DataFrame({
    "Emotion"    : EMOTION_NAMES,
    "BERT"       : bert_f1_per_class.round(3),
    "Gemini (0S)": gemini_f1_per_class.round(3),
    "Δ (BERT−LLM)": diff_f1.round(3),
}).sort_values("BERT", ascending=False).reset_index(drop=True)

print(df_f1.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 10))

heatmap_data = df_f1[["BERT", "Gemini (0S)"]].values  # (28, 2)

sns.heatmap(
    heatmap_data,
    ax=ax,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    cmap="RdYlGn",
    vmin=0.0, vmax=1.0,
    xticklabels=["BERT", "Gemini\nZero-shot"],
    yticklabels=df_f1["Emotion"].tolist(),
    linewidths=0.4,
    linecolor="white",
)
ax.set_title("Per-class F1 Score\n(BERT-base vs Gemini Zero-shot)", fontsize=12, pad=10)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "per_class_f1_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {PLOTS_DIR / 'per_class_f1_heatmap.png'}")

## 4. Disagreement Analysis

In [ ]:
# Disagreement: at least one label slot differs between BERT and Gemini
disagree_mask = (bert_preds_aligned != gemini_preds_aligned).any(axis=1)
n_disagree    = disagree_mask.sum()
pct_disagree  = 100 * n_disagree / N_shared

print(f"Samples where BERT and Gemini disagree: {n_disagree} / {N_shared} ({pct_disagree:.1f}%)")

# Per-class disagreement rate
class_disagree = (bert_preds_aligned != gemini_preds_aligned).mean(axis=0)
disagree_df = pd.DataFrame({
    "Emotion": EMOTION_NAMES,
    "Disagree rate": class_disagree.round(3),
}).sort_values("Disagree rate", ascending=False)

print("\nTop-10 classes by disagreement rate:")
print(disagree_df.head(10).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
top_df = disagree_df.head(15)
ax.barh(top_df["Emotion"][::-1], top_df["Disagree rate"][::-1], color="coral", edgecolor="white")
ax.set_xlabel("Fraction of samples where models disagree")
ax.set_title("Top-15 Classes by BERT vs Gemini Disagreement Rate")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "disagreement_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Error Categorisation — FP / FN per Class

In [ ]:
def count_fp_fn(preds, true):
    """Return per-class FP and FN counts."""
    fp = ((preds == 1) & (true == 0)).sum(axis=0)  # predict positive but ground-truth negative
    fn = ((preds == 0) & (true == 1)).sum(axis=0)  # predict negative but ground-truth positive
    return fp, fn

bert_fp,   bert_fn   = count_fp_fn(bert_preds_aligned,   true_aligned)
gemini_fp, gemini_fn = count_fp_fn(gemini_preds_aligned, true_aligned)

error_df = pd.DataFrame({
    "Emotion"       : EMOTION_NAMES,
    "BERT FP"       : bert_fp,
    "BERT FN"       : bert_fn,
    "Gemini FP"     : gemini_fp,
    "Gemini FN"     : gemini_fn,
}).sort_values("BERT FN", ascending=False)

print("Top-10 classes by BERT False Negatives:")
print(error_df.head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

top_fn_df = error_df.sort_values("BERT FN", ascending=False).head(15)

for ax, model, fp_col, fn_col in [
    (axes[0], "BERT",   "BERT FP",   "BERT FN"),
    (axes[1], "Gemini", "Gemini FP", "Gemini FN"),
]:
    emotions = top_fn_df["Emotion"].tolist()[::-1]
    fp_vals  = top_fn_df[fp_col].tolist()[::-1]
    fn_vals  = [-v for v in top_fn_df[fn_col].tolist()[::-1]]  # negative for left side

    ax.barh(emotions, fn_vals, color="salmon",   label="FN (missed)")
    ax.barh(emotions, fp_vals, color="steelblue", label="FP (over-predicted)")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("← FN  |  FP →")
    ax.set_title(f"{model} — FP / FN per Class")
    ax.legend(fontsize=9)

plt.suptitle("Error Categorisation (top-15 classes by BERT FN)", fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "fp_fn_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Top-20 Most Confused Emotion Pairs

In [ ]:
# For BERT: count co-occurrence of (true label i, predicted label j) where i ≠ j
# i.e., true_i=1 but pred_i=0, AND pred_j=1 (model confused i → j)
confusion_pairs = defaultdict(int)

for sample_true, sample_pred in zip(true_aligned, bert_preds_aligned):
    missed   = np.where((sample_true == 1) & (sample_pred == 0))[0]  # FN: should have predicted
    over_pred = np.where((sample_true == 0) & (sample_pred == 1))[0]  # FP: should NOT have predicted
    # Record each (missed_true, over_predicted) pair as a confusion
    for t_idx in missed:
        for p_idx in over_pred:
            pair = (EMOTION_NAMES[t_idx], EMOTION_NAMES[p_idx])
            confusion_pairs[pair] += 1

top_pairs = sorted(confusion_pairs.items(), key=lambda x: x[1], reverse=True)[:20]

print("Top-20 confused pairs (BERT): true → predicted")
print(f"{'True Emotion':<20} {'Pred Emotion':<20} {'Count':>6}")
print("-" * 50)
for (true_e, pred_e), cnt in top_pairs:
    print(f"{true_e:<20} {pred_e:<20} {cnt:>6}")

In [ ]:
# Pivot to matrix for heatmap
# Only show emotions that appear in the top-20 pairs
involved = sorted(set(e for pair, _ in top_pairs for e in pair))
mat = pd.DataFrame(0, index=involved, columns=involved)
for (true_e, pred_e), cnt in confusion_pairs.items():
    if true_e in mat.index and pred_e in mat.columns:
        mat.loc[true_e, pred_e] = cnt

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    mat,
    ax=ax,
    annot=True, fmt="d", annot_kws={"size": 8},
    cmap="YlOrRd",
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"shrink": 0.7},
)
ax.set_xlabel("Predicted emotion (FP)", fontsize=11)
ax.set_ylabel("True emotion (FN)", fontsize=11)
ax.set_title("BERT Confusion Matrix — Top Co-occurring Error Pairs", fontsize=12, pad=10)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "confusion_pairs_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Qualitative Examples

In [ ]:
from sklearn.metrics import f1_score as sk_f1

def sample_f1(pred_row, true_row):
    """Per-sample F1 (treating the 28-dim vector as one sample)."""
    if true_row.sum() == 0 and pred_row.sum() == 0:
        return 1.0
    if true_row.sum() == 0 or pred_row.sum() == 0:
        return 0.0
    tp = ((pred_row == 1) & (true_row == 1)).sum()
    fp = ((pred_row == 1) & (true_row == 0)).sum()
    fn = ((pred_row == 0) & (true_row == 1)).sum()
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

bert_sample_f1   = np.array([sample_f1(bert_preds_aligned[i],   true_aligned[i]) for i in range(N_shared)])
gemini_sample_f1 = np.array([sample_f1(gemini_preds_aligned[i], true_aligned[i]) for i in range(N_shared)])

# BERT correct (F1=1), Gemini wrong (F1<0.5)
bert_right_gemini_wrong = np.where((bert_sample_f1 == 1.0) & (gemini_sample_f1 < 0.5))[0]
# BERT wrong (F1<0.5), Gemini correct (F1=1)
bert_wrong_gemini_right = np.where((bert_sample_f1 < 0.5) & (gemini_sample_f1 == 1.0))[0]

print(f"BERT correct, Gemini wrong: {len(bert_right_gemini_wrong)} samples")
print(f"BERT wrong, Gemini correct: {len(bert_wrong_gemini_right)} samples")

In [ ]:
def show_examples(indices, label="", n=10):
    print(f"\n{'='*70}")
    print(f"{label}  ({min(n, len(indices))} examples)")
    print(f"{'='*70}")
    for i, idx in enumerate(indices[:n]):
        true_e    = [EMOTION_NAMES[j] for j in range(NUM_LABELS) if true_aligned[idx, j] == 1]
        bert_e    = [EMOTION_NAMES[j] for j in range(NUM_LABELS) if bert_preds_aligned[idx, j] == 1]
        gemini_e  = [EMOTION_NAMES[j] for j in range(NUM_LABELS) if gemini_preds_aligned[idx, j] == 1]
        text      = texts_aligned[idx]
        print(f"[{i+1}] {text[:120]}")
        print(f"     True   : {true_e}")
        print(f"     BERT   : {bert_e}")
        print(f"     Gemini : {gemini_e}")
        print()

show_examples(
    bert_right_gemini_wrong[:10],
    label="✅ BERT correct — ❌ Gemini wrong"
)

In [ ]:
show_examples(
    bert_wrong_gemini_right[:10],
    label="❌ BERT wrong — ✅ Gemini correct"
)

## Summary Statistics

In [ ]:
from sklearn.metrics import f1_score, hamming_loss

bert_f1macro   = f1_score(true_aligned, bert_preds_aligned,   average="macro",  zero_division=0)
bert_f1micro   = f1_score(true_aligned, bert_preds_aligned,   average="micro",  zero_division=0)
bert_hl        = hamming_loss(true_aligned, bert_preds_aligned)
gemini_f1macro = f1_score(true_aligned, gemini_preds_aligned, average="macro",  zero_division=0)
gemini_f1micro = f1_score(true_aligned, gemini_preds_aligned, average="micro",  zero_division=0)
gemini_hl      = hamming_loss(true_aligned, gemini_preds_aligned)

print("="*55)
print(f"{'Metric':<20} {'BERT':>10} {'Gemini (0S)':>14}")
print("-"*55)
print(f"{'F1-macro':<20} {bert_f1macro:>10.4f} {gemini_f1macro:>14.4f}")
print(f"{'F1-micro':<20} {bert_f1micro:>10.4f} {gemini_f1micro:>14.4f}")
print(f"{'Hamming Loss':<20} {bert_hl:>10.4f} {gemini_hl:>14.4f}")
print(f"{'BERT✓ Gemini✗':<20} {len(bert_right_gemini_wrong):>10,}")
print(f"{'BERT✗ Gemini✓':<20} {len(bert_wrong_gemini_right):>14,}")
print("="*55)